# Workflow example for reading and combining CSV files using pandas

We will read all CSV files from a directory, combine them into a single DataFrame, perform some statistics and then save the combined DataFrame to a new CSV file.



In [1]:
# we need to load pandas
import pandas as pd
print("Pandas loaded successfully.")

Pandas loaded successfully.


In [8]:
# get list of all csv files that start with grades_
# we will use Path from pathlib
from pathlib import Path
# let's use rglob
# csv_files = list(Path('.').rglob('grades_*.csv')) # . means current folder, base folder
csv_files = list(Path('.').glob('grades_*.csv')) # i want only local files
print(f"Found {len(csv_files)} CSV files.")

Found 2 CSV files.


In [9]:
# print csv_files
for file in csv_files:
    print(file)

grades_feb.csv
grades_jan.csv


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# if I have a folder in My google drive called StudentuDati I can see its contents as follows:
all_files = list(Path('/content/drive/MyDrive/StudentuDati').rglob('*.csv'))
print(f"Found {len(all_files)} CSV files.")

Found 2 CSV files.


In [5]:
# print those file names
for file in all_files:
    print(file)

/content/drive/MyDrive/StudentuDati/grades_feb.csv
/content/drive/MyDrive/StudentuDati/grades_jan.csv


In [10]:
# let's load them all as a list of dataframes
dataframes = [] # start with empty list
for file in csv_files:
    print(f"Loading {file}...")
    df = pd.read_csv(file)
    # now we add to a list of separate dataframes
    dataframes.append(df)
print(f"Loaded {len(dataframes)} dataframes.")
# could be 0 could be many many dataframes

Loading grades_feb.csv...
Loading grades_jan.csv...
Loaded 2 dataframes.


In [11]:
# let's check that dataframes have same columns
# we can use sets for that
# sets in Python guarantee unique values
first_columns = set(dataframes[0].columns)
all_same = all(set(df.columns) == first_columns for df in dataframes)
print(f"All dataframes have the same columns: {all_same}")
# if not, we cannot combine them directly
if not all_same:
    raise ValueError("Dataframes have different columns and cannot be combined directly.")
# we could have skipped this cell if we already knew we had the same columns

All dataframes have the same columns: True


In [12]:
# now we simply concatenate them
combined_df = pd.concat(dataframes, ignore_index=True)
# print information about combined dataframe
print(f"Combined dataframe has {combined_df.shape[0]} rows and {combined_df.shape[1]} columns.")

Combined dataframe has 60 rows and 5 columns.


In [13]:
# what columns do we have?
print("Columns in combined dataframe:", combined_df.columns.tolist())

Columns in combined dataframe: ['student_id', 'student_name', 'course', 'grade', 'month']


In [14]:
# what unique course we have?
print("Unique courses in combined dataframe:", combined_df['course'].unique())

Unique courses in combined dataframe: ['Math' 'Physics' 'Biology']


In [15]:
# how many grades each course have?
print("Grades per course in combined dataframe:")
print(combined_df['course'].value_counts())

Grades per course in combined dataframe:
course
Math       20
Physics    20
Biology    20
Name: count, dtype: int64


In [16]:
# let's get some statistics
# we want to get grades by course
# we can use group by to aggregate by one or more columns
# then perform some aggregation function like mean, sum, count, etc.
grades_by_course = combined_df.groupby('course')['grade'].mean()
print(grades_by_course)

course
Biology    76.60
Math       77.75
Physics    80.15
Name: grade, dtype: float64


In [17]:
# let's just get some general statistics on combined dataframe
combined_df.describe()

,student_id,grade
count,60.00000,60.000000
mean,5.50000,78.166667
std,2.89652,12.036902
min,1.00000,60.000000
25%,3.00000,67.000000
50%,5.50000,77.000000
75%,8.00000,87.250000
max,10.00000,100.000000


In [18]:
# so let's aggregate by name and course and describe
combined_df.groupby(['student_name', 'course'])['grade'].describe()

count  mean        std   min    25%   50%    75%    max
student_name course                                                          
Alice        Biology    2.0  82.5   3.535534  80.0  81.25  82.5  83.75   85.0
             Math       2.0  73.5  13.435029  64.0  68.75  73.5  78.25   83.0
             Physics    2.0  73.0   4.242641  70.0  71.50  73.0  74.50   76.0
Bob          Biology    2.0  74.0  15.556349  63.0  68.50  74.0  79.50   85.0
             Math       2.0  72.5   0.707107  72.0  72.25  72.5  72.75   73.0
             Physics    2.0  82.5   7.778175  77.0  79.75  82.5  85.25   88.0
Carol        Biology    2.0  76.5  23.334524  60.0  68.25  76.5  84.75   93.0
             Math       2.0  67.5   9.192388  61.0  64.25  67.5  70.75   74.0
             Physics    2.0  84.0   8.485281  78.0  81.00  84.0  87.00   90.0
Dave         Biology    2.0  79.5  24.748737  62.0  70.75  79.5  88.25   97.0
             Math       2.0  75.5  12.020815  67.0  71.25  75.5  79.75   84.0
             Physics    2.0  98.5   0.707107  98.0  98.25  98.5  98.75   99.0
Eve          Biology    2.0  65.0   2.828427  63.0  64.00  65.0  66.00   67.0
             Math       2.0  75.5  16.263456  64.0  69.75  75.5  81.25   87.0
             Physics    2.0  77.5  10.606602  70.0  73.75  77.5  81.25   85.0
Frank        Biology    2.0  73.5   4.949747  70.0  71.75  73.5  75.25   77.0
             Math       2.0  82.0   8.485281  76.0  79.00  82.0  85.00   88.0
             Physics    2.0  70.5  13.435029  61.0  65.75  70.5  75.25   80.0
Grace        Biology    2.0  92.5   0.707107  92.0  92.25  92.5  92.75   93.0
             Math       2.0  80.0   4.242641  77.0  78.50  80.0  81.50   83.0
             Physics    2.0  82.0  25.455844  64.0  73.00  82.0  91.00  100.0
Heidi        Biology    2.0  74.5  10.606602  67.0  70.75  74.5  78.25   82.0
             Math       2.0  90.0   8.485281  84.0  87.00  90.0  93.00   96.0
             Physics    2.0  81.5   6.363961  77.0  79.25  81.5  83.75   86.0
Ivan         Biology    2.0  79.0  25.455844  61.0  70.00  79.0  88.00   97.0
             Math       2.0  76.0  21.213203  61.0  68.50  76.0  83.50   91.0
             Physics    2.0  83.5   7.778175  78.0  80.75  83.5  86.25   89.0
Judy         Biology    2.0  69.0  11.313708  61.0  65.00  69.0  73.00   77.0
             Math       2.0  85.0  21.213203  70.0  77.50  85.0  92.50  100.0
             Physics    2.0  68.5   4.949747  65.0  66.75  68.5  70.25   72.0

In [19]:
# and let's save the aggregated statistics to a new excel file
# first save to new dataframe
aggregated_stats = combined_df.groupby(['student_name', 'course'])['grade'].describe().reset_index()
# now save to excel
aggregated_stats.to_excel('aggregated_grades_statistics.xlsx', index=False)
print("Aggregated statistics saved to 'aggregated_grades_statistics.xlsx'.")

Aggregated statistics saved to 'aggregated_grades_statistics.xlsx'.


In [20]:
# let's download aggregrate statistics to local computer
from google.colab import files
files.download('aggregated_grades_statistics.xlsx') # must match existing file name in the google colab server

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# documentation on Pandas groupby
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html